# 🤖 Modelo de Machine Learning para Predicción de Ventas

Este cuaderno Jupyter detalla el proceso completo de modelado predictivo para la **Pregunta 7**:
1. **Generación de Datos Sintéticos**: Diseñamos un histórico de ventas realista (500 registros) con patrones de precios y descuentos según volumen.
2. **Ingeniería de Features**: Codificación One-Hot de variables categóricas.
3. **Entrenamiento**: Implementación de un modelo de regresión no lineal (**Random Forest Regressor**).
4. **Evaluación**: Medición de métricas R² y RMSE.
5. **Persistencia**: Exportación del modelo entrenado a un archivo `.pkl` para su consumo interactivo en **Streamlit**.

## 1. Importación de Librerías y Semilla

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Fijar semilla para reproducibilidad
np.random.seed(42)

## 2. Generación de Datos de Ventas Históricas

Generaremos 500 registros con 5 productos comunes, donde el precio unitario tiene una pequeña variación aleatoria y se aplican descuentos inteligentes si la cantidad supera ciertos límites (por ejemplo, compras al por mayor).

In [ ]:
n_samples = 500

productos = ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares']
precios_promedio = {
    'Laptop': 2500.0,
    'Mouse': 50.0,
    'Teclado': 120.0,
    'Monitor': 350.0,
    'Auriculares': 80.0
}

# Crear listas vacías para los datos
lista_prod = np.random.choice(productos, size=n_samples)
lista_cant = np.random.randint(1, 11, size=n_samples) # Cantidad entre 1 y 10
lista_prec = []
lista_total = []

for prod, cant in zip(lista_prod, lista_cant):
    # Precio unitario con pequeña variabilidad (+- 5%)
    base_price = precios_promedio[prod]
    unit_price = round(base_price * np.random.uniform(0.95, 1.05), 2)
    lista_prec.append(unit_price)
    
    # Calcular el total aplicando un descuento por volumen
    subtotal = unit_price * cant
    if cant >= 8:
        descuento = 0.15 # 15% de descuento por compra de 8 o más unidades
    elif cant >= 5:
        descuento = 0.08 # 8% de descuento por compra de 5 a 7 unidades
    else:
        descuento = 0.0 # Sin descuento
        
    lista_total.append(round(subtotal * (1 - descuento), 2))

# Construir DataFrame
df_historico = pd.DataFrame({
    'id_venta': range(1, n_samples + 1),
    'producto': lista_prod,
    'cantidad': lista_cant,
    'precio': lista_prec,
    'total_venta': lista_total
})

# Guardar el CSV
df_historico.to_csv('ventas_historicas.csv', index=False)
print("¡Dataset histórico generado con éxito!")
df_historico.head()

## 3. Preparación de Variables (One-Hot Encoding)

Dado que los modelos matemáticos no entienden texto de manera directa, codificamos la variable categórica `producto` usando One-Hot Encoding.

In [ ]:
# Codificar variables categóricas
df_encoded = pd.get_dummies(df_historico, columns=['producto'], dtype=int)

# Separar features (X) y target (y)
# Eliminamos id_venta (no tiene valor predictivo) y total_venta (nuestro target)
X = df_encoded.drop(columns=['id_venta', 'total_venta'])
y = df_encoded['total_venta']

# Guardar el orden de las columnas de entrenamiento por seguridad en producción
columnas_modelo = list(X.columns)
print("Variables de entrada para el modelo:", columnas_modelo)
X.head()

## 4. Partición de Datos y Entrenamiento del Modelo

Dividimos los datos en Train (80%) y Test (20%) para evaluar el rendimiento general del modelo predictivo.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicializar el Regresor Random Forest
# Usaremos un bosque con 100 árboles de decisión en paralelo
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print("¡Modelo entrenado exitosamente!")

## 5. Evaluación de Métricas

In [ ]:
# Predicciones sobre el conjunto de test
y_pred = rf_model.predict(X_test)

# Calcular métricas estadísticas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Métricas del Modelo:")
print(f"  - Coeficiente R²: {r2:.4f} (Excelente poder predictivo)")
print(f"  - Error Cuadrático Medio (RMSE): ${rmse:.2f}")

## 6. Persistencia y Exportación del Modelo

Guardamos en un diccionario compacto el modelo entrenado, la lista exacta de columnas de entrenamiento y el mapeo de variables categóricas para su posterior carga en Streamlit de forma fluida.

In [ ]:
model_data = {
    'model': rf_model,
    'features': columnas_modelo,
    'productos_validos': productos,
    'precios_promedio': precios_promedio
}

joblib.dump(model_data, 'modelo_ventas.pkl')
print("¡Binario 'modelo_ventas.pkl' exportado correctamente!")